In [ ]:
%load_ext autoreload
%autoreload 2

import shutil
import sys
from pathlib import Path

from ultralytics import YOLO

# Paths come from path_config.yaml; edit that file's current_workstation to switch machines.
from path_config import PMOF_CODE_DIR, DATA_BASE_DIR, DATA_ACTIONCLS_DIR

if not PMOF_CODE_DIR.is_dir():
    raise FileNotFoundError(f"PMOF code directory not found: {PMOF_CODE_DIR}")

sys.path.insert(0, str(PMOF_CODE_DIR))

from src.data import (
    imgid_to_annpath,
    imgid_to_imgpath,
    list_record_ids,
    read_annotation,
    recordid_to_imageids,
)


In [ ]:
record_ids = list_record_ids()
record_ids

In [ ]:
train_record_ids = ["rec4", "rec22", "rec25", "rec29", "rec30"]
val_record_ids = ["rec27"]
test_record_ids = ["rec28"]

# Restructure
Doc for Dataset Structure:  https://docs.ultralytics.com/datasets/classify#folder-structure-example

In [ ]:
#make directories
PMOF_actioncls_base_dir = Path(DATA_ACTIONCLS_DIR)

for split in ["train", "val", "test"]:
    for cls in ["seated", "other"]:
        (PMOF_actioncls_base_dir / split / cls).mkdir(parents=True, exist_ok=True)

In [ ]:
def copy_images_by_action(
    record_ids: list[str],
    output_dir: Path,
    split: str,
    seated_action: str = "seated",
) -> int:
    """Copy PMOF images into an Ultralytics classification folder layout.

    Creates:
        output_dir/
        ├── {split}/
        │   ├── seated/
        │   └── other/

    An image is classified as 'seated' only if every annotated person has
    action == seated_action; if any person has another action, it is
    classified as 'other'.

    Returns the number of images processed.
    """
    output_dir = Path(output_dir)
    seated_dir = output_dir / split / "seated"
    other_dir = output_dir / split / "other"
    seated_dir.mkdir(parents=True, exist_ok=True)
    other_dir.mkdir(parents=True, exist_ok=True)

    images_counter = 0
    for record_id in record_ids:
        for image_id in recordid_to_imageids(record_id):
            images_counter += 1

            imgpath = Path(imgid_to_imgpath(image_id))
            if not imgpath.is_file():
                print(f"Warning: image not found: {imgpath}")
                continue

            annpath = imgid_to_annpath(image_id)
            person_anns = [
                ann for ann in read_annotation(annpath, image_id)
                if ann.category_name == "person"
            ]
            has_other_action = any(ann.action != seated_action for ann in person_anns)

            destination_dir = other_dir if has_other_action else seated_dir
            shutil.copy2(imgpath, destination_dir / imgpath.name)

    print(f"{split}: copied {images_counter} images")
    return images_counter


In [ ]:
for split, split_record_ids in [("val", val_record_ids), ("train", train_record_ids), ("test", test_record_ids)]:
    copy_images_by_action(split_record_ids, PMOF_actioncls_base_dir, split)


# Train Model

In [ ]:
# Load a pretrained model (recommended for training)
import torch
import torchvision.transforms as T

from ultralytics import YOLO
from ultralytics.data.dataset import ClassificationDataset
from ultralytics.models.yolo.classify import ClassificationTrainer, ClassificationValidator

class CustomizedDataset(ClassificationDataset):
    """A customized dataset class for image classification with enhanced data augmentation transforms."""

    def __init__(self, root: str, args, augment: bool = False, prefix: str = ""):
        """Initialize a customized classification dataset with enhanced data augmentation transforms."""
        super().__init__(root, args, augment, prefix)

        # Add your custom training transforms here
        train_transforms = T.Compose(
            [
                T.Resize((args.imgsz, args.imgsz)),
                T.RandomHorizontalFlip(p=args.fliplr),
                T.RandomVerticalFlip(p=args.flipud),
                #T.RandAugment(interpolation=T.InterpolationMode.BILINEAR),
                T.ColorJitter(brightness=args.hsv_v, contrast=args.hsv_v, saturation=args.hsv_s, hue=args.hsv_h),
                T.ToTensor(),
                T.Normalize(mean=torch.tensor(0), std=torch.tensor(1)),
                T.RandomErasing(p=args.erasing, inplace=True),
            ]
        )

        # Add your custom validation transforms here
        val_transforms = T.Compose(
            [
                T.Resize((args.imgsz, args.imgsz)),
                T.ToTensor(),
                T.Normalize(mean=torch.tensor(0), std=torch.tensor(1)),
            ]
        )
        self.torch_transforms = train_transforms if augment else val_transforms

class CustomizedTrainer(ClassificationTrainer):
    """A customized trainer class for YOLO classification models with enhanced dataset handling."""

    def build_dataset(self, img_path: str, mode: str = "train", batch=None):
        """Build a customized dataset for classification training and the validation during training."""
        return CustomizedDataset(root=img_path, args=self.args, augment=mode == "train", prefix=mode)

class CustomizedValidator(ClassificationValidator):
    """A customized validator class for YOLO classification models with enhanced dataset handling."""

    def build_dataset(self, img_path: str):
        """Build a customized dataset for classification standalone validation (no augmentation)."""
        return CustomizedDataset(root=img_path, args=self.args, augment=False, prefix=self.args.split)



model = YOLO("yolo26s-cls.pt")

model.train(data=str(PMOF_actioncls_base_dir), trainer=CustomizedTrainer, epochs=100, patience=5, imgsz=320, batch=64)
model.val(data=str(PMOF_actioncls_base_dir), validator=CustomizedValidator, imgsz=320, batch=64)
model.val(data=str(PMOF_actioncls_base_dir), validator=CustomizedValidator, imgsz=320, batch=64, split="val")

In [ ]:
model.val(data=str(PMOF_actioncls_base_dir), validator=CustomizedValidator, imgsz=320, batch=64, split="test")

In [ ]:
model.predict(source=str(PMOF_actioncls_base_dir / "val/other/rec27_001889.png"), visualize=True, save=True)